# Review counts per language on the canonical 6.2M-user base

Out-of-core adaptation of `language_distribution_full.ipynb`. The target population is the set of users matchable to a public profile (about 6.2M). For each such user, reviews are counted per language.

In [ ]:
from pathlib import Path
import os
import time

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
REVIEWS_DATA = ROOT / 'data' / 'corpus' / 'reviews_by_lang'
USER_FEATURES_FILE = ROOT / 'data' / 'features' / 'user_features.parquet'
PARTS_DIR = ROOT / 'data' / 'features' / 'language_counts_6m_parts'
OUTPUT_FILE = ROOT / 'data' / 'features' / 'en_users_language_counts_6m.parquet'
RESULTS_DIR = ROOT / 'results' / 'language_reviews_6m'
TEMP_DIR = ROOT / 'data' / 'tmp_duckdb_language_reviews'
REBUILD = False  # True forces the artifacts to be rebuilt

for directory in [PARTS_DIR, RESULTS_DIR, TEMP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def sql_path(path):
    return str(Path(path).resolve()).replace('\\', '/').replace("'", "''")

con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='6GB'")
con.execute(f"SET temp_directory='{sql_path(TEMP_DIR)}'")
con.execute("SET preserve_insertion_order=false")
print('DuckDB', duckdb.__version__)
print('User source:', USER_FEATURES_FILE)
print('Fonte de reviews:', REVIEWS_DATA)

## 1. Target population and available languages

The canonical key is `user_key` (SteamID64). In the reviews it is extracted only from URLs of the form `.../profiles/<SteamID64>`.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE target_users AS
    SELECT DISTINCT CAST(user_key AS VARCHAR) AS user_key
    FROM read_parquet('{sql_path(USER_FEATURES_FILE)}')
    WHERE user_key IS NOT NULL
""")
n_target = con.execute('SELECT count(*) FROM target_users').fetchone()[0]
lang_dirs = sorted(p for p in REVIEWS_DATA.glob('review_lang=*') if p.is_dir())
langs = [p.name.split('=', 1)[1] for p in lang_dirs]

print(f'Target users: {n_target:,}')
print(f'{len(langs)} languages: {langs}')
assert n_target == 6_204_110, f'Expected 6,204,110 users; found {n_target:,}'
assert 'en' in langs

## 2. Per-language counting (out-of-core and resumable)

Each language produces a small Parquet with `(user_key, lang, n_reviews)`. Files already finished are reused, so an interrupted run resumes where it stopped.

In [ ]:
t_all = time.time()
part_rows = []

for lang_dir, lang in zip(lang_dirs, langs):
    out_file = PARTS_DIR / f'lang={lang}.parquet'
    tmp_file = PARTS_DIR / f'lang={lang}.tmp.parquet'
    if REBUILD:
        out_file.unlink(missing_ok=True)
    tmp_file.unlink(missing_ok=True)

    t0 = time.time()
    if not out_file.exists():
        source_glob = lang_dir / '*.parquet'
        lang_sql = lang.replace("'", "''")
        con.execute(f"""
            COPY (
                SELECT t.user_key, '{lang_sql}'::VARCHAR AS lang, count(*)::INTEGER AS n_reviews
                FROM (
                    SELECT nullif(regexp_extract(user_url, 'profiles/([0-9]+)', 1), '') AS user_key
                    FROM read_parquet('{sql_path(source_glob)}', union_by_name=true)
                ) r
                INNER JOIN target_users t USING (user_key)
                WHERE r.user_key IS NOT NULL
                GROUP BY t.user_key
            ) TO '{sql_path(tmp_file)}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
        os.replace(tmp_file, out_file)
        status = 'built'
    else:
        status = 'reused'

    n_rows, n_reviews = con.execute(
        f"SELECT count(*), coalesce(sum(n_reviews), 0) FROM read_parquet('{sql_path(out_file)}')"
    ).fetchone()
    part_rows.append((lang, n_rows, n_reviews))
    print(f'{lang:>8}: {n_rows:>10,} users | {n_reviews:>12,} reviews | {status} | {time.time()-t0:6.1f}s')

parts_summary = pd.DataFrame(part_rows, columns=['lang', 'users', 'reviews'])
print(f'\nTempo da etapa: {(time.time()-t_all)/60:.1f} min')
parts_summary

## 3. Consolidation into one row per user

`lang_counts` is stored as `MAP(VARCHAR, INTEGER)`. The derived columns `total_reviews`, `n_languages`, and `en_reviews` are also written.

In [ ]:
tmp_output = OUTPUT_FILE.with_suffix('.tmp.parquet')
if REBUILD:
    OUTPUT_FILE.unlink(missing_ok=True)
tmp_output.unlink(missing_ok=True)

if not OUTPUT_FILE.exists():
    t0 = time.time()
    con.execute(f"""
        COPY (
            SELECT
                user_key,
                map(list(lang ORDER BY lang), list(n_reviews ORDER BY lang)) AS lang_counts,
                sum(n_reviews)::BIGINT AS total_reviews,
                count(*)::INTEGER AS n_languages,
                max(CASE WHEN lang='en' THEN n_reviews ELSE 0 END)::BIGINT AS en_reviews
            FROM read_parquet('{sql_path(PARTS_DIR / 'lang=*.parquet')}')
            GROUP BY user_key
        ) TO '{sql_path(tmp_output)}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    os.replace(tmp_output, OUTPUT_FILE)
    print(f'Consolidated built in {(time.time()-t0)/60:.1f} min')
else:
    print('Reused existing consolidated file')

print('Resultado:', OUTPUT_FILE)
print(f'Tamanho: {OUTPUT_FILE.stat().st_size / 1024**2:,.1f} MiB')
con.execute(f"SELECT * FROM read_parquet('{sql_path(OUTPUT_FILE)}') LIMIT 5").fetchdf()

## 4. Full validation and independent recomputation

In [ ]:
validation = con.execute(f"""
    SELECT
        count(*) AS rows,
        count(DISTINCT user_key) AS unique_users,
        count(*) FILTER (WHERE user_key IS NULL) AS null_user_key,
        count(*) FILTER (WHERE en_reviews < 1) AS missing_en,
        count(*) FILTER (WHERE total_reviews < en_reviews) AS invalid_totals,
        min(n_languages) AS min_languages,
        max(n_languages) AS max_languages,
        sum(total_reviews)::HUGEINT AS reviews_in_output
    FROM read_parquet('{sql_path(OUTPUT_FILE)}')
""").fetchdf()
reviews_in_parts = con.execute(
    f"SELECT sum(n_reviews)::HUGEINT FROM read_parquet('{sql_path(PARTS_DIR / 'lang=*.parquet')}')"
).fetchone()[0]
display(validation)

v = validation.iloc[0]
assert int(v['rows']) == n_target
assert int(v['unique_users']) == n_target
assert int(v['null_user_key']) == 0
assert int(v['missing_en']) == 0
assert int(v['invalid_totals']) == 0
assert int(v['reviews_in_output']) == int(reviews_in_parts)
print('Full validation: OK')

sample_users = con.execute(f"""
    SELECT user_key
    FROM read_parquet('{sql_path(OUTPUT_FILE)}')
    WHERE n_languages >= 3
    ORDER BY hash(user_key)
    LIMIT 5
""").fetchdf()
con.register('sample_users_df', sample_users)
recomputed = con.execute(f"""
    SELECT s.user_key, r.review_lang AS lang, count(*)::BIGINT AS n_reviews
    FROM (
        SELECT
            nullif(regexp_extract(user_url, 'profiles/([0-9]+)', 1), '') AS user_key,
            review_lang
        FROM read_parquet('{sql_path(REVIEWS_DATA / 'review_lang=*' / '*.parquet')}',
                          hive_partitioning=true, union_by_name=true)
    ) r
    INNER JOIN sample_users_df s USING (user_key)
    GROUP BY s.user_key, r.review_lang
    ORDER BY s.user_key, r.review_lang
""").fetchdf()
saved = con.execute(f"""
    SELECT p.user_key, p.lang, p.n_reviews::BIGINT AS n_reviews
    FROM read_parquet('{sql_path(PARTS_DIR / 'lang=*.parquet')}') p
    INNER JOIN sample_users_df s USING (user_key)
    ORDER BY p.user_key, p.lang
""").fetchdf()
pd.testing.assert_frame_equal(recomputed, saved)
print('Independent sample recomputation: OK')
saved

## 5. Descriptive statistics

In [ ]:
summary = con.execute(f"""
    SELECT
        count(*) AS users,
        count(*) FILTER (WHERE n_languages=1) AS only_en,
        count(*) FILTER (WHERE n_languages>=2) AS multilingual,
        avg(n_languages) AS mean_languages,
        median(n_languages) AS median_languages,
        max(n_languages) AS max_languages,
        avg(total_reviews) AS mean_reviews,
        median(total_reviews) AS median_reviews,
        quantile_cont(total_reviews, 0.90) AS p90_reviews,
        max(total_reviews) AS max_reviews,
        avg(100.0 * en_reviews / total_reviews) AS mean_pct_en,
        median(100.0 * en_reviews / total_reviews) AS median_pct_en,
        100.0 * avg((en_reviews=total_reviews)::INTEGER) AS pct_users_100_en,
        100.0 * avg((100.0*en_reviews/total_reviews < 50)::INTEGER) AS pct_users_lt50_en
    FROM read_parquet('{sql_path(OUTPUT_FILE)}')
""").fetchdf()
display(summary.T.rename(columns={0: 'value'}))

lang_distribution = con.execute(f"""
    SELECT n_languages, count(*) AS users
    FROM read_parquet('{sql_path(OUTPUT_FILE)}')
    GROUP BY n_languages ORDER BY n_languages
""").fetchdf()
lang_distribution

## 6. CDFs

In [ ]:
cdf_data = con.execute(f"""
    SELECT n_languages, 100.0 * en_reviews / total_reviews AS pct_en
    FROM read_parquet('{sql_path(OUTPUT_FILE)}')
""").fetchnumpy()

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'pdf.fonttype': 42,  # TrueType embutido; linhas e eixos permanecem vetoriais
    'ps.fonttype': 42,
})

x = np.sort(cdf_data['n_languages'])
y = np.arange(1, len(x)+1) / len(x)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y, linewidth=2)
ax.set(xlabel='Number of languages used per user', ylabel='CDF')
ax.set_xticks(range(1, int(x.max())+1))
ax.grid(True, color='lightgray')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'cdf_languages_per_user.png', dpi=180)
fig.savefig(RESULTS_DIR / 'cdf_languages_per_user.pdf', format='pdf', bbox_inches='tight')
plt.show()

x2 = np.sort(cdf_data['pct_en'])
y2 = np.arange(1, len(x2)+1) / len(x2)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x2, y2, linewidth=2)
ax.set(xlabel='% of reviews written in English (per user)', ylabel='CDF', xlim=(0, 100))
ax.grid(True, color='lightgray')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'cdf_pct_english_per_user.png', dpi=180)
fig.savefig(RESULTS_DIR / 'cdf_pct_english_per_user.pdf', format='pdf', bbox_inches='tight')
plt.show()

print('Figures saved to:', RESULTS_DIR)